In [1]:
import os
os.chdir('/Users/jordancassella/QTSFinalProject/QTSFinalProject/Drafts')
import numpy as np 
import pandas as pd 
import processing_module as pr 

ModuleNotFoundError: No module named 'processing_module'

In [2]:
os.chdir('/Users/jordancassella/QTSFinalProject/QTSFinalProject')
raw_data = pd.read_csv('all_features_all_tokens.csv').drop('Unnamed: 0', axis=1)
raw_data.head()

,metricKey,defaultValue,createdAt,symbol
0,staking_ratio,74.03726,2021-03-01T01:40:05Z,ADA
1,staking_ratio,74.03726,2021-03-01T02:00:00Z,ADA
2,staking_ratio,74.03726,2021-03-01T03:40:04Z,ADA
3,staking_ratio,74.03726,2021-03-01T04:00:00Z,ADA
4,staking_ratio,74.03726,2021-03-01T05:40:05Z,ADA


In [3]:
unique_metrics = raw_data['metricKey'].unique()
unique_coins = raw_data['symbol'].unique()
print(f'Number of unique staking metrics: {len(unique_metrics)}')
print(unique_metrics)
print(f'Number of unique tokens: {len(unique_coins)}')
print(unique_coins)

Number of unique staking metrics: 6
['staking_ratio' 'total_staking_wallets' 'active_validators'
 'annualized_rewards_usd' 'real_reward_rate' 'staked_tokens']
Number of unique tokens: 6
['ADA' 'DOT' 'SOL' 'NEAR' 'AVAX' 'MATIC']


In [4]:
processed_data = pr.process_staking_data(raw_data)
for token, df in processed_data.items():
    num_nans = df.isna().sum().sum()
    print(f'Number of NAs: {num_nans} for {token}')

Number of NAs: 19029 for ADA
Number of NAs: 12638 for DOT
Number of NAs: 16305 for SOL
Number of NAs: 34890 for NEAR
Number of NAs: 3973 for AVAX
Number of NAs: 2696 for MATIC


In [5]:
processed_data['NEAR'].iloc[15000:]

metricKey,active_validators,annualized_rewards_usd,real_reward_rate,staked_tokens,staking_ratio,total_staking_wallets
createdAt_rounded,,,,,,
2022-11-16 03:00:00+00:00,117.0,9.161826e+07,NaN,4.683332e+08,42.299235,85378.0
2022-11-16 04:00:00+00:00,117.0,9.161826e+07,NaN,4.683332e+08,42.299235,85378.0
2022-11-16 05:00:00+00:00,117.0,9.254370e+07,NaN,4.683332e+08,42.299236,85390.0
2022-11-16 06:00:00+00:00,117.0,9.254370e+07,NaN,4.683332e+08,42.299236,85390.0
2022-11-16 07:00:00+00:00,117.0,9.210852e+07,NaN,4.683332e+08,42.299237,85467.0
...,...,...,...,...,...,...
2025-03-03 19:00:00+00:00,233.0,1.805225e+08,4.050210,6.021670e+08,48.634344,260940.0
2025-03-03 20:00:00+00:00,233.0,1.805225e+08,4.050210,6.021670e+08,48.634344,260940.0
2025-03-03 21:00:00+00:00,233.0,1.654789e+08,4.050208,6.021670e+08,48.634355,260945.0


In [6]:
### number of parameters in 129-64-32-3 NN: 104350. We would need 10 times that as the number of samples.
### If we have 35000 samples (full staking_df), a 64-32-16-3 NN would work, since this yields 26590 parameters.
### This is the '10 times rule'
### in MLP_Practice, we are shifting the features up one, instead of shifting the returns up one, introducing lookahead bias

In [7]:
processed_data['AVAX'].iloc[480:].to_parquet('avax_staking_data.parquet')

In [8]:
avax = processed_data['AVAX'].iloc[480:]

In [9]:
pr.check_missing_hours(avax)

DatetimeIndex(['2021-04-10 02:00:00+00:00', '2023-06-19 01:00:00+00:00',
               '2023-06-19 02:00:00+00:00', '2023-06-19 03:00:00+00:00',
               '2023-06-19 04:00:00+00:00', '2023-06-19 05:00:00+00:00',
               '2023-06-19 06:00:00+00:00', '2023-06-19 07:00:00+00:00',
               '2023-06-19 08:00:00+00:00', '2023-08-18 02:00:00+00:00',
               ...
               '2023-08-22 04:00:00+00:00', '2023-08-22 05:00:00+00:00',
               '2023-08-22 06:00:00+00:00', '2023-08-22 07:00:00+00:00',
               '2023-08-22 08:00:00+00:00', '2023-08-22 09:00:00+00:00',
               '2023-08-22 10:00:00+00:00', '2023-08-22 11:00:00+00:00',
               '2023-08-22 12:00:00+00:00', '2023-11-06 01:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=117, freq=None)

In [10]:
avax_filled = avax.fillna(method='ffill')
avax_filled['createdAt_rounded'] = avax_filled.index 
av = pr.preprocess_data(data=avax_filled, set_index=True, staking_data=True)

In [11]:
av.isna().any().any()

False

In [13]:
av.to_parquet('avax_staking_data2.parquet')

In [15]:
av

,active_validators,real_reward_rate,staked_tokens,staking_ratio,total_staking_wallets
2021-03-21 02:00:00+00:00,889.0,0.693402,3.018003e+08,79.023851,6253.0
2021-03-21 03:00:00+00:00,889.0,0.693402,3.018003e+08,79.023851,6253.0
2021-03-21 04:00:00+00:00,889.0,0.693368,3.018010e+08,79.024018,6258.0
2021-03-21 05:00:00+00:00,889.0,0.693368,3.018010e+08,79.024018,6258.0
2021-03-21 06:00:00+00:00,889.0,0.693643,3.017842e+08,79.019616,6260.0
...,...,...,...,...,...
2025-03-03 20:00:00+00:00,1418.0,3.281419,2.284129e+08,50.160244,42283.0
2025-03-03 21:00:00+00:00,1418.0,3.281419,2.284129e+08,50.160244,42283.0
2025-03-03 22:00:00+00:00,1419.0,3.283371,2.283060e+08,50.136733,42302.0
2025-03-03 23:00:00+00:00,1419.0,3.283371,2.283060e+08,50.136733,42302.0


In [21]:
# Number of NANs
processed_data['MATIC'].isna().sum()

metricKey
active_validators           0
real_reward_rate         2696
staked_tokens               0
staking_ratio               0
total_staking_wallets       0
dtype: int64

In [22]:
matic = processed_data['MATIC'].drop('real_reward_rate', axis = 1)
matic.isna().any().any()

False